# conv2d_chord_only.py -- training job (RunPod)

Runs 6-fold player-held-out cross-validation for `Conv2dChordOnlyModel` (2D CNN backbone, chord-identity classification -- 42 chord classes + NO_CHORD, nothing about string/fret).

**Before running:** working directory must be the project root on the pod (e.g. `/workspace`) -- needs `foundational_heads.py`, `conv2d_chord_only.py`, `preprocessing/` (including `precompute_chord_annotations.py`), `data/guitarset/annotation/`, and `output/processed_cqt/` already present. Unlike the pitch/tab notebooks, you do **not** need to upload a chord label cache -- this notebook builds `output/chord_annotations/` directly on the pod from files already there (fast: no audio decoding, just reads existing CQT shapes + parses `.jams` text).

**Kernel:** whichever default kernel RunPod's Jupyter starts with -- if it's a PyTorch/CUDA template, torch is already installed system-wide.

In [ ]:
import os

# Uncomment and edit if this notebook isn't already running from the project root:
# os.chdir("/workspace")

print("cwd:", os.getcwd())
for required in ["foundational_heads.py", "conv2d_chord_only.py",
                  "preprocessing/data_splits.py", "preprocessing/precompute_chord_annotations.py",
                  "data/guitarset/annotation", "output/processed_cqt"]:
    status = "OK" if os.path.exists(required) else "MISSING"
    print(f"  [{status}] {required}")

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")
if device.type != "cuda":
    print("WARNING: not using CUDA -- if this is a RunPod GPU pod, training will be much slower than expected. Check the pod's PyTorch/CUDA install.")

## Build the chord label cache (if it's not already there)
Runs `preprocessing/precompute_chord_annotations.py` directly on the pod. Idempotent-ish -- safe to re-run, but skipped automatically if `output/chord_annotations/` already looks complete (360 `.npz` files + the vocab file).

In [ ]:
chord_cache_dir = "output/chord_annotations"
existing = len([f for f in os.listdir(chord_cache_dir) if f.endswith(".npz")]) if os.path.isdir(chord_cache_dir) else 0

if existing >= 360 and os.path.exists(os.path.join(chord_cache_dir, "_vocab.json")):
    print(f"Chord cache already present ({existing} files) -- skipping precompute.")
else:
    print(f"Chord cache incomplete ({existing}/360) -- running precompute...")
    !python preprocessing/precompute_chord_annotations.py

In [ ]:
import json

from conv2d_chord_only import Conv2dChordOnlyModel, cross_validate_by_player_chord
from foundational_heads import find_cqt_jams_pairs

with open("output/chord_annotations/_vocab.json") as f:
    vocab = json.load(f)
num_classes = len(vocab)
print(f"{num_classes} chord classes: {vocab}")

## Hyperparameters
Same values currently in `conv2d_chord_only.py`'s `main()` -- edit here to experiment without touching the .py file.

In [ ]:
input_bins = 252
sr = 22050
hop_length = 512
time_frames = round(4 * sr / hop_length)   # 4s sliding window
stride = round(3 * sr / hop_length)        # 3s stride -> 1s overlap
batch_size = 16
temporal_hidden = 256
latent_dim = 128
dropout = 0.4

epochs = 100
lr = 0.001
patience = 10

In [ ]:
cqt_files, jams_files = find_cqt_jams_pairs()
print(f"{len(cqt_files)} debleeded recordings paired")

In [ ]:
def build_model():
    return Conv2dChordOnlyModel(
        input_bins=input_bins,
        num_classes=num_classes,
        temporal_hidden=temporal_hidden,
        latent_dim=latent_dim,
        dropout=dropout,
    )

# quick sanity check before committing to the full run
_m = build_model().to(device)
_x = torch.randn(2, 1, time_frames, input_bins).to(device)
_out = _m(_x)
print("forward pass OK, output shape:", tuple(_out.shape))
print("params:", sum(p.numel() for p in _m.parameters()))
del _m, _x, _out

## Run the job
Trains 6 models (one per held-out player). Progress prints per-epoch. Per-fold checkpoints saved to `output/conv2d_chord_only_checkpoint_holdout{player}.pt`.

**Don't close the browser tab while this runs** -- the kernel keeps running on the pod regardless; only stopping/terminating the *pod* actually kills it.

In [ ]:
results = cross_validate_by_player_chord(
    model_fn=build_model,
    cqt_files=cqt_files,
    jams_files=jams_files,
    time_frames=time_frames,
    stride=stride,
    batch_size=batch_size,
    epochs=epochs,
    lr=lr,
    patience=patience,
    device=device,
    checkpoint_dir="output",
    checkpoint_prefix="conv2d_chord_only_checkpoint",
    extra_checkpoint_fields={
        "input_bins": input_bins,
        "num_classes": num_classes,
        "temporal_hidden": temporal_hidden,
        "latent_dim": latent_dim,
        "dropout": dropout,
    },
)
print("Cross-validation job successful.")

## Inspect results
`results` is a list of per-fold metric dicts (`held_out_player`, `loss`, `accuracy`, `baseline`) -- `accuracy` is chord classification accuracy, `baseline` is the trivial "always predict the most common chord" accuracy to judge it against.

In [ ]:
import pandas as pd
pd.DataFrame(results)